In [ ]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary
from metasmith.python_api import Resources, Size, Duration

dtypes, containers, transforms = Std()
base_dir = Path("./cache")

path_to_agent_home = (base_dir/"local_home").resolve()
smith = Agent(
    home = Source.FromLocal(path_to_agent_home),
)
smith.Deploy()

In [ ]:
for k in dtypes.types:
    if "accession" not in k: continue
    print(k)

In [ ]:
for k in dtypes.types:
    if "annotations" not in k: continue
    print(k)

In [ ]:
in_dir = base_dir/"std_assembly_data.xgdb"
inputs = DataInstanceLibrary(in_dir)
# inputs.AddItem("./lr_ss10.fastq", "std::long_reads")
# inputs.AddItem("./cb2a.acc", "std::long_reads_accession")
inputs.AddItem("./epi300.acc", "std::long_reads_accession")
# inputs.AddItem("./SRR36161492.acc", "std::long_reads_accession")
inputs.Save()

In [ ]:
for x in inputs.Iterate():
    print(x)

In [ ]:

task = smith.GenerateWorkflow(
    samples=[inputs],
    resources=[containers],
    transforms=[transforms],
    targets=[
        # dtypes["short_reads_assembly"],
        # dtypes["assembly"],
        dtypes["busco_annotations"],
        # dtypes["hybrid_assembly"],
        # dtypes["bakta_annotations"].WithLineage([dtypes["hybrid_assembly"]]),
        # dtypes["functional_annotations"].WithLineage([dtypes["long_reads_assembly"]]),
        # dtypes["functional_annotations"].WithLineage([dtypes["hybrid_assembly"]]),
        # dtypes["per_contig_coverage"].WithLineage([dtypes["short_reads_assembly"]]),
    ],
    max_refine=256,
)
# for s in task.plans[0][0].steps:
#     print(s.order, s.transform.name)
#     for u in s.uses:
#         print(f"  {u.dtype.key} {u.dtype_name}")
#     print()

sum(len(p.steps) for g in task.plans for p in g)

In [ ]:
from IPython.display import Image
base_dir = Path("./cache")
dagf = Path(base_dir / f"dag_std_assembly")
task.plans[0][0].RenderDAG(dagf, format="png")
Image(filename=f"{dagf}.png")

In [ ]:
smith.StageWorkflow(task, on_exist="clear")

In [ ]:
for path, tr in transforms.IterateTransforms():
    print(tr.name)

In [ ]:
smith.RunWorkflow(
    task,
    config_file=smith.GetNxfConfigPresets()["local"],
    resource_overrides={
        "all": Resources(
            cpus=8,
        ),
        transforms["busco_ref"]: Resources(
            cpus=2,
        ),
        transforms["fasterq_long"]: Resources(
            cpus=2,
        ),
    }
)